
# GAEZ v5 suitability and attainable yield downloader

## Description

This notebook was created to simplify the downloading of **GAEZ v5 suitability and attainable yield raster datasets** from the FAO Google Cloud Storage repository.

It uses the official RES05 README file as a metadata catalog to identify available datasets and build the correct download links. Users can select a variable, period, climate data source, SSP, crop, and water/input management level from interactive controls, compile the matching files, and download the selected rasters locally.

This notebook is fully self-contained. It does **not** use an external `utilities.py` file. All imports, functions, catalog loading, filtering, and download logic are included directly in the notebook.


## Install packages if needed

In [1]:
# Uncomment if needed
# !pip install pandas openpyxl requests ipywidgets


## Imports, settings, and utility functions

In [2]:

from pathlib import Path
from urllib.parse import quote
from io import BytesIO
import time
import warnings

import pandas as pd
import requests

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------

README_URL = "https://data.apps.fao.org/catalog/dataset/514c92d5-9e01-4c70-814a-80ea1ca9fe6a/resource/768b08ad-be9c-427a-84be-3a3d6dd7835a/download/_readme_res05.xlsx"

BASE_URL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET"

DEFAULT_OUT_DIR = Path("downloads_res05")
REQUEST_TIMEOUT = 30
DOWNLOAD_TIMEOUT = 120
CHUNK_SIZE = 1024 * 1024

DEFAULT_OUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Catalog loading
# ------------------------------------------------------------------

def download_readme_from_url(readme_url=README_URL):
    """Download the RES05 README Excel file from the FAO URL and return it as BytesIO."""
    response = requests.get(readme_url, timeout=DOWNLOAD_TIMEOUT)
    response.raise_for_status()
    return BytesIO(response.content)


def clean_code_columns(df):
    """Clean column names and blank string values."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
            df.loc[df[col].isin(["nan", "None", ""]), col] = pd.NA
    return df.dropna(how="all")


def clean_header(value):
    """Clean a header cell from the README."""
    if pd.isna(value):
        return ""
    return str(value).strip().upper().replace(" ", "_")


def read_list_files_sheet(excel_bytes):
    """Read the LIST_FILES sheet even if the header is not on the first row."""
    raw = pd.read_excel(excel_bytes, sheet_name="LIST_FILES", header=None, dtype=str)

    expected_columns = [
        "MAPSET_CODE",
        "PERIOD",
        "CLIMATE_DATA_SOURCE",
        "SSP",
        "CROP",
        "INPUT",
        "FILE_NAME",
    ]

    header_row = None
    best_score = 0

    for i in range(min(40, len(raw))):
        row_values = [clean_header(v) for v in raw.iloc[i].tolist()]
        score = sum(col in row_values for col in expected_columns)
        if score > best_score:
            best_score = score
            header_row = i

    if header_row is None or best_score < 4:
        raise ValueError("Could not find the header row in the LIST_FILES sheet.")

    columns = [clean_header(v) for v in raw.iloc[header_row].tolist()]
    catalog = raw.iloc[header_row + 1:].copy()
    catalog.columns = columns
    catalog = catalog.loc[:, [c for c in catalog.columns if c != ""]]

    missing = [c for c in expected_columns if c not in catalog.columns]
    if missing:
        raise ValueError(f"Missing expected columns in LIST_FILES: {missing}")

    catalog = catalog[expected_columns].copy()
    catalog = clean_code_columns(catalog)
    catalog = catalog.dropna(subset=["FILE_NAME"])

    return catalog.reset_index(drop=True)


def build_url(row, base_url=BASE_URL):
    """Build the public cloud URL from MAPSET_CODE and FILE_NAME."""
    mapset_code = str(row["MAPSET_CODE"]).strip()
    file_name = str(row["FILE_NAME"]).strip()

    # Some README sheets include FILE_NAME without .tif.
    if not file_name.lower().endswith(".tif"):
        download_name = file_name + ".tif"
    else:
        download_name = file_name

    return f"{base_url}/{quote(mapset_code)}/{quote(download_name)}"


def read_code_sheet(excel_bytes, sheet_name, code_col="CODE", label_col=None):
    """Read one CODES_* sheet into a dictionary of code -> label."""
    df = pd.read_excel(excel_bytes, sheet_name=sheet_name, dtype=str)
    df = clean_code_columns(df)

    if code_col not in df.columns:
        # Fallback: use first column as code
        code_col = df.columns[0]

    if label_col is None or label_col not in df.columns:
        possible = [c for c in df.columns if c != code_col]
        if not possible:
            return {}
        label_col = possible[0]

    df = df.dropna(subset=[code_col])
    return dict(zip(df[code_col].astype(str), df[label_col].astype(str)))


def load_catalog_and_codes(readme_url=README_URL, base_url=BASE_URL):
    """Load the RES05 LIST_FILES catalog and code dictionaries from the online README."""
    excel_content = download_readme_from_url(readme_url)

    # Read LIST_FILES first
    catalog = read_list_files_sheet(excel_content)

    # Re-create BytesIO before each group of reads
    excel_content = download_readme_from_url(readme_url)

    catalog["DOWNLOAD_NAME"] = catalog["FILE_NAME"].astype(str).str.strip()
    catalog.loc[~catalog["DOWNLOAD_NAME"].str.lower().str.endswith(".tif"), "DOWNLOAD_NAME"] = (
        catalog.loc[~catalog["DOWNLOAD_NAME"].str.lower().str.endswith(".tif"), "DOWNLOAD_NAME"] + ".tif"
    )

    catalog["DOWNLOAD_URL"] = catalog.apply(lambda row: build_url(row, base_url=base_url), axis=1)

    # Read code sheets
    excel_content = download_readme_from_url(readme_url)

    code_maps = {
        "variable": read_code_sheet(excel_content, "CODES_VARIABLE", "CODE", "VARIABLE"),
    }

    excel_content = download_readme_from_url(readme_url)
    code_maps["period"] = read_code_sheet(excel_content, "CODES_PERIOD", "CODE", "PERIOD")

    excel_content = download_readme_from_url(readme_url)
    code_maps["climate"] = read_code_sheet(excel_content, "CODES_CLIMATE_DATA_SOURCE", "CODE", "CLIMATE_DATA_SOURCE")

    excel_content = download_readme_from_url(readme_url)
    code_maps["ssp"] = read_code_sheet(excel_content, "CODES_SSP", "CODE", "SSP")

    excel_content = download_readme_from_url(readme_url)
    code_maps["crop"] = read_code_sheet(excel_content, "CODES_CROP", "CODE", "Crop name")

    excel_content = download_readme_from_url(readme_url)
    code_maps["input"] = read_code_sheet(excel_content, "CODES_WATER_INPUT", "CODE", "WATER CONTENT AND INPUT MANAGEMENT LEVEL")

    return {
        "readme_url": readme_url,
        "catalog": catalog,
        "code_maps": code_maps,
        "variable_dict": code_maps["variable"],
        "period_dict": code_maps["period"],
        "climate_dict": code_maps["climate"],
        "ssp_dict": code_maps["ssp"],
        "crop_dict": code_maps["crop"],
        "input_dict": code_maps["input"],
    }


# ------------------------------------------------------------------
# Filtering and download helpers
# ------------------------------------------------------------------

def label_options(values, label_map=None, include_all=False, all_label="All", all_value="__ALL__"):
    """Create labelled dropdown options."""
    clean = [
        v for v in pd.Series(values).dropna().astype(str).unique().tolist()
        if v and v != "nan"
    ]
    clean = sorted(clean)

    options = []

    if include_all:
        options.append((all_label, all_value))

    for value in clean:
        label = f"{value} — {label_map.get(value, value)}" if label_map else value
        options.append((label, value))

    return options


def period_options(values, period_map):
    """Create period options with grouped HP/FP choices."""
    opts = [
        ("All HP historical periods", "__ALL_HP__"),
        ("All FP future periods", "__ALL_FP__"),
    ]
    opts.extend(label_options(values, period_map))
    return opts


def resolve_period_selection(selection, available_periods):
    """Resolve period dropdown selection into a list of period codes."""
    available = pd.Series(available_periods).dropna().astype(str).unique().tolist()

    if selection == "__ALL_HP__":
        return [p for p in available if p.startswith("HP")]

    if selection == "__ALL_FP__":
        return [p for p in available if p.startswith("FP")]

    if selection == "__ALL__":
        return available

    return [selection]


def normalize_selection(value):
    """Return a clean list for filtering, or None to mean all values."""
    if value is None:
        return None

    if isinstance(value, str):
        if value in ("", "__ALL__"):
            return None
        return [value]

    if isinstance(value, (list, tuple, set, pd.Series)):
        values = list(value)
    elif hasattr(value, "tolist"):
        values = value.tolist()
        if not isinstance(values, list):
            values = [values]
    else:
        values = [value]

    values = [
        str(v) for v in values
        if pd.notna(v) and str(v) not in ("", "__ALL__")
    ]

    return values or None


def filter_catalog(catalog, variable=None, periods=None, climates=None, ssps=None, crops=None, inputs=None):
    """Filter catalog by selected dimensions. Use None for all values."""
    df = catalog.copy()

    filters = {
        "MAPSET_CODE": normalize_selection(variable),
        "PERIOD": normalize_selection(periods),
        "CLIMATE_DATA_SOURCE": normalize_selection(climates),
        "SSP": normalize_selection(ssps),
        "CROP": normalize_selection(crops),
        "INPUT": normalize_selection(inputs),
    }

    for col, vals in filters.items():
        if vals is not None:
            df = df[df[col].isin(vals)]

    return df.reset_index(drop=True)


def compile_selection(catalog, variable, periods, crops, climates=None, ssps=None, inputs=None, max_crops=3):
    """Compile selected combination(s). Crops must contain 1 to max_crops crops."""
    crops = list(crops or [])

    if len(crops) < 1 or len(crops) > max_crops:
        raise ValueError(f"Please select between 1 and {max_crops} crops.")

    return filter_catalog(
        catalog,
        variable=variable,
        periods=periods,
        climates=climates,
        ssps=ssps,
        crops=crops,
        inputs=inputs,
    )


def compile_all_for_variable(catalog, variable, periods=None, climates=None, ssps=None, inputs=None):
    """Compile all datasets for one variable, with optional period/climate/SSP/input filters."""
    return filter_catalog(
        catalog,
        variable=variable,
        periods=periods,
        climates=climates,
        ssps=ssps,
        inputs=inputs,
    )


def head_exists(url, timeout=8):
    """Check whether a URL exists."""
    try:
        r = requests.head(url, timeout=timeout, allow_redirects=True)
        return r.status_code == 200
    except Exception:
        return False


def add_exists_column(df, enabled=False, max_checks=200):
    """Add an EXISTS column using HTTP HEAD checks."""
    df = df.copy()

    if not enabled or df.empty:
        return df

    if len(df) > max_checks:
        df["EXISTS"] = pd.NA
        return df

    df["EXISTS"] = [head_exists(u) for u in df["DOWNLOAD_URL"]]

    return df


def download_row(row, out_dir=DEFAULT_OUT_DIR, overwrite=False):
    """Download one catalog row."""
    out_dir = Path(out_dir)

    url = row["DOWNLOAD_URL"]
    variable = str(row["MAPSET_CODE"])
    period = str(row["PERIOD"])
    climate = str(row["CLIMATE_DATA_SOURCE"])
    ssp = str(row["SSP"])
    input_code = str(row["INPUT"])

    folder = out_dir / variable / f"{period}_{climate}_{ssp}_{input_code}"
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / row["DOWNLOAD_NAME"]

    if out_path.exists() and not overwrite:
        return out_path, "skipped_exists"

    with requests.get(url, stream=True, timeout=DOWNLOAD_TIMEOUT) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)

    return out_path, "downloaded"


def download_rows(df, out_dir=DEFAULT_OUT_DIR, overwrite=False, sleep_seconds=0.05):
    """Download all rows in a compiled dataframe."""
    paths = []
    statuses = []

    for _, row in df.iterrows():
        path, status = download_row(row, out_dir=out_dir, overwrite=overwrite)
        paths.append(path)
        statuses.append(status)
        time.sleep(sleep_seconds)

    return pd.DataFrame({
        "path": [str(p) for p in paths],
        "status": statuses,
    })


def download_selected_row(row, out_dir=DEFAULT_OUT_DIR, overwrite=False):
    """Download one selected row."""
    return download_row(row, out_dir=out_dir, overwrite=overwrite)


def download_all_for_variable(df, out_dir=DEFAULT_OUT_DIR, overwrite=False):
    """Download all rows for the current compiled dataframe."""
    return download_rows(df, out_dir=out_dir, overwrite=overwrite)


## Load the RES05 README catalog

In [3]:

data = load_catalog_and_codes()

catalog = data["catalog"]
code_maps = data["code_maps"]

print("README source:")
print(data["readme_url"])
print()
print("Number of available files:", len(catalog))
display(catalog.head())


README source:
https://data.apps.fao.org/catalog/dataset/514c92d5-9e01-4c70-814a-80ea1ca9fe6a/resource/768b08ad-be9c-427a-84be-3a3d6dd7835a/download/_readme_res05.xlsx

Number of available files: 109056


,MAPSET_CODE,PERIOD,CLIMATE_DATA_SOURCE,SSP,CROP,INPUT,FILE_NAME,DOWNLOAD_NAME,DOWNLOAD_URL
0,RES05-ETL,HP0120,AGERA5,HIST,ALF,HILM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.HILM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.HILM.tif,https://storage.googleapis.com/fao-gismgr-gaez...
1,RES05-ETL,HP0120,AGERA5,HIST,ALF,HRLM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.HRLM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.HRLM.tif,https://storage.googleapis.com/fao-gismgr-gaez...
2,RES05-ETL,HP0120,AGERA5,HIST,ALF,LILM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.LILM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.LILM.tif,https://storage.googleapis.com/fao-gismgr-gaez...
3,RES05-ETL,HP0120,AGERA5,HIST,ALF,LRLM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.LRLM,GAEZ-V5.RES05-ETL.HP0120.AGERA5.HIST.ALF.LRLM.tif,https://storage.googleapis.com/fao-gismgr-gaez...
4,RES05-ETL,HP8100,AGERA5,HIST,ALF,HILM,GAEZ-V5.RES05-ETL.HP8100.AGERA5.HIST.ALF.HILM,GAEZ-V5.RES05-ETL.HP8100.AGERA5.HIST.ALF.HILM.tif,https://storage.googleapis.com/fao-gismgr-gaez...


## Downloader Selector

In [4]:

def launch_res05_downloader(catalog, code_maps, out_dir=DEFAULT_OUT_DIR):
    """Display the simplified RES05 downloader interface."""
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    mode_dd = widgets.ToggleButtons(
        options=[
            ("Download selected combination(s)", "selection"),
            ("Download all datasets for one variable", "all_variable"),
        ],
        description="Mode:",
        layout=widgets.Layout(width="900px"),
    )

    variable_dd = widgets.Dropdown(description="Variable:", layout=widgets.Layout(width="760px"))
    period_dd = widgets.Dropdown(description="Period:", layout=widgets.Layout(width="760px"))
    climate_dd = widgets.Dropdown(description="Climate:", layout=widgets.Layout(width="760px"))
    ssp_dd = widgets.Dropdown(description="SSP:", layout=widgets.Layout(width="760px"))
    input_dd = widgets.Dropdown(description="Input:", layout=widgets.Layout(width="760px"))

    crop_ms = widgets.SelectMultiple(
        description="Crop(s):",
        options=[],
        rows=10,
        layout=widgets.Layout(width="760px"),
    )

    check_exists_cb = widgets.Checkbox(
        value=False,
        description="Check cloud file existence during compile (slower; skipped automatically for >200 rows)",
        layout=widgets.Layout(width="900px"),
    )

    overwrite_cb = widgets.Checkbox(
        value=False,
        description="Overwrite existing downloaded files",
        layout=widgets.Layout(width="900px"),
    )

    compile_btn = widgets.Button(
        description="Compile selection",
        button_style="info",
        icon="check",
    )

    download_btn = widgets.Button(
        description="Download compiled files",
        button_style="success",
        icon="download",
    )

    file_ms = widgets.SelectMultiple(
        description="Compiled files:",
        options=[],
        rows=12,
        layout=widgets.Layout(width="1120px"),
    )

    download_scope = widgets.RadioButtons(
        options=[
            ("Download all compiled rows", "all_compiled"),
            ("Download only selected rows from the file list", "selected_files"),
        ],
        value="all_compiled",
        description="Download:",
        layout=widgets.Layout(width="760px"),
    )

    status_out = widgets.Output()
    result_out = widgets.Output()
    ui_out = widgets.Output()

    state = {"compiled_df": pd.DataFrame()}

    def set_dropdown_value(dropdown, preferred=None):
        values = [v for _, v in dropdown.options]

        if preferred in values:
            dropdown.value = preferred
        elif dropdown.value in values:
            return
        elif values:
            dropdown.value = values[0]
        else:
            dropdown.value = None

    def current_periods(base_df):
        return resolve_period_selection(period_dd.value, base_df["PERIOD"])

    def update_dropdowns(*args):
        old_values = {
            "variable": variable_dd.value,
            "period": period_dd.value,
            "climate": climate_dd.value,
            "ssp": ssp_dd.value,
            "input": input_dd.value,
            "crops": tuple(crop_ms.value),
        }

        variable_dd.options = label_options(catalog["MAPSET_CODE"], code_maps["variable"])
        set_dropdown_value(variable_dd, old_values["variable"])

        df_var = filter_catalog(catalog, variable=variable_dd.value)

        period_dd.options = period_options(df_var["PERIOD"], code_maps["period"])
        set_dropdown_value(period_dd, old_values["period"] or "HP0120")

        periods = current_periods(df_var)
        df_period = filter_catalog(df_var, periods=periods)

        climate_dd.options = label_options(
            df_period["CLIMATE_DATA_SOURCE"],
            code_maps["climate"],
            include_all=True,
            all_label="All climate data sources",
        )
        set_dropdown_value(climate_dd, old_values["climate"] or "__ALL__")

        climates = None if climate_dd.value == "__ALL__" else [climate_dd.value]
        df_climate = filter_catalog(df_period, climates=climates)

        ssp_dd.options = label_options(
            df_climate["SSP"],
            code_maps["ssp"],
            include_all=True,
            all_label="All SSPs",
        )
        set_dropdown_value(ssp_dd, old_values["ssp"] or "__ALL__")

        ssps = None if ssp_dd.value == "__ALL__" else [ssp_dd.value]
        df_ssp = filter_catalog(df_climate, ssps=ssps)

        input_dd.options = label_options(
            df_ssp["INPUT"],
            code_maps["input"],
            include_all=True,
            all_label="All water/input levels",
        )
        set_dropdown_value(input_dd, old_values["input"] or "__ALL__")

        inputs = None if input_dd.value == "__ALL__" else [input_dd.value]
        df_input = filter_catalog(df_ssp, inputs=inputs)

        crop_ms.options = label_options(df_input["CROP"], code_maps["crop"])

        available_crops = [v for _, v in crop_ms.options]
        kept = tuple([c for c in old_values["crops"] if c in available_crops])

        if kept:
            crop_ms.value = kept[:3]
        elif "COF" in available_crops:
            crop_ms.value = ("COF",)
        else:
            crop_ms.value = tuple(available_crops[:1])

    def update_mode_ui(*args):
        with ui_out:
            clear_output()

            base_items = [
                widgets.HTML("<h3>GAEZ v5 suitability and attainable yield downloader</h3>"),
                widgets.HTML("Use the controls below to compile available GAEZ datasets from the RES05 README catalog and download them from Google Cloud Storage."),
                mode_dd,
                variable_dd,
                period_dd,
                climate_dd,
                ssp_dd,
                input_dd,
            ]

            if mode_dd.value == "selection":
                base_items.extend([
                    crop_ms,
                    widgets.HTML("<b>Note:</b> select 1–3 crops for this mode."),
                ])

            base_items.extend([
                check_exists_cb,
                widgets.HBox([compile_btn, download_btn]),
                overwrite_cb,
                download_scope,
                file_ms,
                status_out,
                result_out,
            ])

            display(widgets.VBox(base_items))

    def build_current_matches():
        periods = resolve_period_selection(
            period_dd.value,
            filter_catalog(catalog, variable=variable_dd.value)["PERIOD"],
        )

        climates = None if climate_dd.value == "__ALL__" else [climate_dd.value]
        ssps = None if ssp_dd.value == "__ALL__" else [ssp_dd.value]
        inputs = None if input_dd.value == "__ALL__" else [input_dd.value]

        if mode_dd.value == "selection":
            matches = compile_selection(
                catalog,
                variable=variable_dd.value,
                periods=periods,
                climates=climates,
                ssps=ssps,
                crops=list(crop_ms.value),
                inputs=inputs,
                max_crops=3,
            )
        else:
            matches = compile_all_for_variable(
                catalog,
                variable=variable_dd.value,
                periods=periods,
                climates=climates,
                ssps=ssps,
                inputs=inputs,
            )

        return add_exists_column(matches, enabled=check_exists_cb.value)

    def on_compile_clicked(b):
        with status_out:
            clear_output()
            print("Compiling selection...")

        with result_out:
            clear_output()

            try:
                state["compiled_df"] = build_current_matches()
            except Exception as e:
                state["compiled_df"] = pd.DataFrame()
                file_ms.options = []
                with status_out:
                    clear_output()
                    print("Compile error:", e)
                return

            compiled_df = state["compiled_df"]

            if compiled_df.empty:
                file_ms.options = []
                print("No matching files found for the selected filters.")
                with status_out:
                    clear_output()
                    print("No matches.")
                return

            show_cols = [
                "MAPSET_CODE",
                "PERIOD",
                "CLIMATE_DATA_SOURCE",
                "SSP",
                "CROP",
                "INPUT",
                "DOWNLOAD_NAME",
                "DOWNLOAD_URL",
            ]

            if "EXISTS" in compiled_df.columns:
                show_cols.append("EXISTS")

            display(compiled_df[show_cols])

            file_options = [
                (name, name) for name in compiled_df["DOWNLOAD_NAME"].tolist()
            ]

            file_ms.options = file_options
            file_ms.value = tuple(
                [name for name, _ in file_options[:min(20, len(file_options))]]
            )

        with status_out:
            clear_output()
            print(f"Compiled {len(state['compiled_df'])} file(s). Review the table, then download.")

    def on_download_clicked(b):
        with status_out:
            clear_output()

            compiled_df = state["compiled_df"]

            if compiled_df.empty:
                print("No compiled files yet. Click 'Compile selection' first.")
                return

            to_download = compiled_df.copy()

            if download_scope.value == "selected_files":
                chosen = list(file_ms.value)

                if not chosen:
                    print("No files selected in the compiled file list.")
                    return

                to_download = to_download[to_download["DOWNLOAD_NAME"].isin(chosen)]

            print(f"Downloading {len(to_download)} file(s)...")

        with result_out:
            try:
                summary = download_rows(
                    to_download,
                    out_dir=out_dir,
                    overwrite=overwrite_cb.value,
                )
                clear_output()
                print(f"Finished. Files saved under: {out_dir.resolve()}")
                display(summary)
            except Exception as e:
                print("Download error:", e)
                print("If this is a 404 error, the file name exists in the README but may not be available at the expected cloud path.")

        with status_out:
            clear_output()
            print("Download step completed. Check the results table below.")

    for dd in [variable_dd, period_dd, climate_dd, ssp_dd, input_dd]:
        dd.observe(update_dropdowns, names="value")

    mode_dd.observe(update_mode_ui, names="value")
    compile_btn.on_click(on_compile_clicked)
    download_btn.on_click(on_download_clicked)

    update_dropdowns()
    update_mode_ui()

    display(ui_out)


In [5]:

launch_res05_downloader(catalog, code_maps, out_dir=DEFAULT_OUT_DIR)


Output()